# AAMem Phase Runner

Run the first cell on Colab. It clones or pulls the latest code from `https://github.com/lenhokhoa23/j4f.git`, then switches the runtime into that folder.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/lenhokhoa23/j4f.git'
COLAB_ROOT = Path('/content')

if COLAB_ROOT.exists():
    PROJECT_DIR = COLAB_ROOT / 'j4f'
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR, check=True)
    elif not PROJECT_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        print(f'{PROJECT_DIR} exists but is not a git repo; using current directory instead.')
        PROJECT_DIR = Path.cwd().resolve()
else:
    PROJECT_DIR = Path.cwd().resolve()

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)


## Setup

Use `MODEL_PRESET='smoke'` for a no-model pipeline check. On Colab GPU, try `qwen2_5_1_5b`, then `qwen2_5_3b`, then `qwen2_5_7b`.

In [ ]:
from pathlib import Path
import json

MODEL_PRESET = 'smoke'
# MODEL_PRESET = 'qwen2_5_1_5b'
# MODEL_PRESET = 'qwen2_5_3b'
# MODEL_PRESET = 'qwen2_5_7b'

from aamem_lab.model_presets import presets_as_dict
presets_as_dict()


## Optional install for HF local models

Skip this for `MODEL_PRESET='smoke'`. Run it for Qwen/Llama/Gemma presets.

In [ ]:
# Uncomment on Colab GPU for HF model runs.
# %pip install -q transformers accelerate


## Phase 2: ActMem answer-level run

Outputs go to `runs/notebook_phase2`. The JSONL contains context, answer, judge, paper metrics, and run config.

In [ ]:
cmd = [
    sys.executable, '-m', 'aamem_lab.phase2_answer_runner',
    '--dataset', 'actmem',
    '--limit', '10',
    '--k', '5',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', str(PROJECT_DIR / 'runs' / 'notebook_phase2'),
    '--tag', f'actmem_n10_k5_{MODEL_PRESET}',
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## Phase 3: STALE-style SR/PR/IPA run

This is our synthetic stale suite aligned to status recognition, premise resistance, and implicit preference/action.

In [ ]:
cmd = [
    sys.executable, '-m', 'aamem_lab.phase3_stale_runner',
    '--limit', '12',
    '--k', '5',
    '--dimensions', 'SR,PR,IPA',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', str(PROJECT_DIR / 'runs' / 'notebook_phase3'),
    '--tag', f'stale_n12_k5_sr_pr_ipa_{MODEL_PRESET}',
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## Read summaries

In [ ]:
for path in sorted((PROJECT_DIR / 'runs').glob('notebook_*/*_summary.json')):
    print('\n===', path, '===')
    print(path.read_text(encoding='utf-8')[:4000])
